# Tray segmentation → full-res warp → HLS (HSL) hue segmentation → grid inference

This notebook:
1. Loads trained YOLOv8 segmentation weights
2. Reads full-resolution phone images from `Images/orig/`
3. Plain-resizes each to 640×640 and saves to `Images/resized/`
4. Runs YOLO on the resized image to get a tray mask
5. Uses the mask (scaled from 640→full-res) to warp the *full-res* image to a top-down tray view (aspect ratio preserved)
6. Applies the requested HSL/HLS segmentation procedure (Hue threshold + mask + AND) on the warped tray

Outputs are written to `out/warped` and `out/segmented` (plus `out/overlay` for quick QC).


In [ ]:
# --- Imports ---
from pathlib import Path
from glob import glob
import numpy as np
import cv2
import math
from ultralytics import YOLO
from scipy.signal import find_peaks


In [ ]:
# Config
WEIGHTS = Path("../models/trayseg_v13.pt")      # trained weights
model = YOLO(WEIGHTS)
# Input/output folders
ORIG_DIR = Path("Images/orig")                # full-resolution phone images
RESIZED_DIR = Path("Images/resized")          # 640x640 plain-resize images (generated)
RESIZED_DIR.mkdir(parents=True, exist_ok=True)

# YOLO inference (runs on resized images)
IMGSZ = 640
CONF = 0.25

# Warping (applies to full-res image using scaled mask corners)
# out_long controls the longer side of the warped tray; aspect ratio is preserved.
WARP_LONG_SIDE = 1400

OUTDIR = Path("out")
(OUTDIR/"warped").mkdir(parents=True, exist_ok=True)
(OUTDIR/"debug").mkdir(parents=True, exist_ok=True)

# Grid inference parameters 
SMOOTH_K = 7            # 1D smoothing window for projections
PEAK_PROM_FRAC = 0.25   # prominence as fraction of max projection
RIM_FRAC = 0.0          # ignore outer rim of tray when detecting grid lines
MIN_CELLS = 2
MAX_CELLS = 200

In [ ]:
# =========================
# Grid inference from dark-texture periodicity (prof-style preproc + projections)
# Grayscale -> threshold (<thr) -> erode/dilate -> projection profiles -> peak detection
# Produces overlay of inferred grid lines on the (warped) tray image.
# =========================
from scipy.signal import find_peaks
from scipy.ndimage import median_filter


def _smooth_1d(x: np.ndarray, k: int) -> np.ndarray:
    """Simple moving-average smoothing for 1D projections."""
    k = int(k)
    if k < 3:
        return x.astype(np.float32)
    if k % 2 == 0:
        k += 1
    ker = np.ones(k, np.float32) / float(k)
    return np.convolve(x.astype(np.float32), ker, mode="same")


def separator_mask_graytray_refined(warped_bgr: np.ndarray, debug_dir=None, debug_prefix: str=None) -> np.ndarray:
    """Return a binary mask (0/255) of likely *grey tray plastic* pixels.

    Uses Lab chroma (near-neutral), HSV saturation (low), and L (reject very dark soil/shadows).
    Designed to suppress green plants + soil, and keep the tray separator ridges.
    """
    lab = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(lab)
    hsv = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2HSV)
    Hh, S, V = cv2.split(hsv)

    a16 = a.astype(np.int16)
    b16 = b.astype(np.int16)
    neutral = np.sqrt((a16 - 128) ** 2 + (b16 - 128) ** 2).astype(np.float32)

    Lf = L.astype(np.float32)
    Sf = S.astype(np.float32)

    # Tray-likeness score: lower is more tray-like.
    # - low chroma (neutral)
    # - low saturation (reject plants + many soils)
    # - moderate lightness (reject very dark soil/shadows)
    score = neutral + 0.6 * Sf + 0.15 * np.abs(Lf - 120.0)

    thr = np.percentile(score, 25)  # keep best 25% tray-like pixels
    mask = (score <= thr).astype(np.uint8) * 255
    mask[neutral > 35] = 0

    # Aggressive rejection of dark regions (soil/shadows/holes)
    mask[L < 70] = 0
    # Reject saturated regions (plants/colorful soil bits)
    mask[S > 60] = 0
    # Reject bright regions (leaves/soil)
    mask[L > 200] = 0

    # Extra vegetation rejection (handles pale / low-sat leaves that slip through)
    Bc, Gc, Rc = cv2.split(warped_bgr)
    Bf = Bc.astype(np.int16)
    Gf2 = Gc.astype(np.int16)
    Rf = Rc.astype(np.int16)

    # Excess Green index: strong for vegetation even when saturation is not huge
    exg = (2 * Gf2 - Rf - Bf).astype(np.int16)
    # Adaptive ExG threshold (more stable across lighting)
    exg_thr = np.percentile(exg, 70)  # tune 70–85
    veg = (exg > exg_thr) & (Gc > Rc + 3) & (Gc > Bc + 3)
    mask[veg] = 0

    # pixels deep inside blobs are likely not separator ridges
    m = (mask > 0).astype(np.uint8)
    dist_in = cv2.distanceTransform(m, cv2.DIST_L2, 3)
    mask[dist_in > 6.0] = 0   # tune 3–10

    soilish = ((b.astype(np.int16) - 128) > 15) & (L > 70)  # tune
    mask[soilish] = 0

    # remove brown pixels
    a16 = a.astype(np.int16)
    b16 = b.astype(np.int16)
    brown = (b16 > 124) & (a16 > 130) & (L > 60)   # tune thresholds
    mask[brown] = 0

    # bright green-ish (including gray-green leaves)
    green_a = (a16 < 124) & (L > 80)     # tune 122–126 and L gate
    mask[green_a] = 0

    # Denoise + cleanup
    mask = median_filter(mask, size=3).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=2)

    # Remove small components (speckle noise)
    num, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    H, W = mask.shape[:2]
    min_area = int(0.00001 * H * W)  # ~0.001% of image

    clean = np.zeros_like(mask)
    for i in range(1, num):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == i] = 255
    mask = clean

    if debug_dir is not None and debug_prefix:
        cv2.imwrite(str(Path(debug_dir) / f"{debug_prefix}_sep_mask_graytray.jpg"), mask)

    return mask


def extract_separator_longlines(bin_mask_u8: np.ndarray, k_frac: float = 0.025, preclose: int = 15):
    """From a binary tray-separator mask (white=tray/separator), build directionally-connected
    horizontal/vertical separator evidence.

    Key difference vs the earlier 'opening' approach:
    - We use directional *closing* to bridge small gaps caused by rounded junctions / occlusions.
    - We do NOT require components to be globally long/continuous (no CC span filtering).

    Returns (lines, horiz, vert, preclosed) all uint8 0/255.
    """
    H, W = bin_mask_u8.shape[:2]
    m = bin_mask_u8.copy()

    # Bridge small gaps in the separator mask (junctions, tiny breaks)
    k = max(3, int(preclose))
    if k % 2 == 0:
        k += 1
    pre = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8), iterations=1)

    # Directional kernels (shorter => less strict, more robust)
    kx = max(15, int(W * float(k_frac)))
    ky = max(15, int(H * float(k_frac)))
    if kx % 2 == 0:
        kx += 1
    if ky % 2 == 0:
        ky += 1

    hker = cv2.getStructuringElement(cv2.MORPH_RECT, (kx, 1))
    vker = cv2.getStructuringElement(cv2.MORPH_RECT, (1, ky))

    # Directional closing: connect separators along the axis
    horiz = cv2.morphologyEx(pre, cv2.MORPH_CLOSE, hker, iterations=1)
    vert  = cv2.morphologyEx(pre, cv2.MORPH_CLOSE, vker, iterations=1)

    # Light opening to suppress big filled regions introduced by closing
    small = np.ones((3, 3), np.uint8)
    horiz = cv2.morphologyEx(horiz, cv2.MORPH_OPEN, small, iterations=1)
    vert  = cv2.morphologyEx(vert,  cv2.MORPH_OPEN, small, iterations=1)

    lines = cv2.bitwise_or(horiz, vert)
    lines = cv2.morphologyEx(lines, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=1)
    return lines, horiz, vert, pre


def _estimate_period_autocorr(sig: np.ndarray, lag_min: int, lag_max: int):
    """Return best period (lag) in [lag_min, lag_max] using normalized autocorrelation."""
    sig = sig.astype(np.float32)
    if sig.size < 8:
        return None
    sig = sig - float(np.mean(sig))
    denom = float(np.dot(sig, sig)) + 1e-6
    ac = np.correlate(sig, sig, mode="full")[sig.size - 1:] / denom  # lag>=0

    lag_min = int(max(1, lag_min))
    lag_max = int(min(lag_max, ac.size - 1))
    if lag_max <= lag_min:
        return None

    window = ac[lag_min:lag_max + 1]
    k = int(np.argmax(window))
    if window[k] < 0.05:
        return None
    return float(lag_min + k)


def infer_grid_from_separators(
    warped_bgr: np.ndarray,
    rim_frac: float = 0.0,
    debug_dir=None,
    debug_prefix: str = None,
    min_period_px: int = 60,
    max_period_px: int | None = None,
):
    """Infer grid using *tray-separator evidence* only.

    Pipeline:
      1) Mask grey tray plastic (suppresses plants/soil).
      2) Build directional 'longlines' evidence (horiz/vert) via directional closing/opening.
      3) Convert to 1D projection profiles (sum along axis).
      4) Find peak locations, estimate the **dominant spacing** (mode of peak-to-peak diffs).
      5) Use that spacing + best phase offset to synthesize a *complete* grid (fills missing lines).

    No RANSAC is used.
    Returns (rows, cols, info_dict, overlay_bgr).
    """
    H, W = warped_bgr.shape[:2]
    x0, y0 = 0, 0
    x1, y1 = W, H

    if max_period_px is None:
        max_period_px = int(max(30, min(W, H) * 0.45))

    tray_mask = separator_mask_graytray_refined(
        warped_bgr, debug_dir=debug_dir, debug_prefix=debug_prefix
    )
    lines, horiz, vert, preclosed = extract_separator_longlines(tray_mask)

    if debug_dir is not None and debug_prefix:
        dd = Path(debug_dir)
        dd.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_preclosed.jpg"), preclosed)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_longlines.jpg"), lines)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_horiz.jpg"), horiz)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_vert.jpg"), vert)


    # Work in inner ROI (avoid outer rim artifacts)
    # NOTE: We intentionally use the *raw* longlines evidence (plus a slight dilation) to avoid
    # losing weak separator signals under plant occlusion.
    lines_u8 = (lines > 127).astype(np.uint8) * 255
    lines_u8 = cv2.dilate(lines_u8, cv2.getStructuringElement(cv2.MORPH_RECT,(3,3)),1)

    # inference mask
    lines_infer = lines_u8.copy()
    border_th = 4

    lines_infer[:border_th,:] = 255
    lines_infer[-border_th:,:] = 255
    lines_infer[:,:border_th] = 255
    lines_infer[:,-border_th:] = 255

    # ROI for 1D periodicity (keeps outer rim out of the projections)
    roi = lines_infer[y0:y1, x0:x1]
    roi_h, roi_w = roi.shape[:2]

    # Projections: separators are bright -> higher sums
    proj_x = (roi > 0).sum(axis=0).astype(np.float32)  # vertical separators -> peaks in X
    proj_y = (roi > 0).sum(axis=1).astype(np.float32)  # horizontal separators -> peaks in Y

    # # Ignore top fraction ONLY for PERIOD ESTIMATION
    # TOP_IGNORE_FRAC = 0.20
    # ignore = int(TOP_IGNORE_FRAC * roi_h)
    # ignore = min(max(ignore, 0), roi_h - 8)  # keep enough samples for autocorr

    # roi_period = roi[ignore:, :]

    # Period-only projections
    # proj_x_period = (roi_period > 0).sum(axis=0).astype(np.float32)  # still len = roi_w
    # proj_y_period = (roi_period > 0).sum(axis=1).astype(np.float32)  # len = roi_h-ignore

    # sx_period = _smooth_1d(proj_x_period, k=max(31, roi_w // 40))
    # sy_period = _smooth_1d(proj_y_period, k=max(31, (roi_h - ignore) // 80))

    sx = _smooth_1d(proj_x, k=max(31, roi_w // 40))
    sy = _smooth_1d(proj_y, k=max(31, roi_h // 80))

    # Estimate dominant pitch via autocorrelation
    per_x = _estimate_period_autocorr(sx, lag_min=min_period_px, lag_max=int(min(max_period_px, roi_w // 2)))
    per_y = _estimate_period_autocorr(sy, lag_min=min_period_px, lag_max=int(min(max_period_px, roi_h // 2)))

    # Anchor peaks (phase): pick leftmost / topmost strong peak
    sxn = (sx - sx.min()) / (np.ptp(sx) + 1e-6)
    syn = (sy - sy.min()) / (np.ptp(sy) + 1e-6)
    dist_x = max(8, int(roi_w / 80))
    dist_y = max(8, int(roi_h / 120))
    px, _ = find_peaks(sxn, distance=dist_x, prominence=0.02)
    py, _ = find_peaks(syn, distance=dist_y, prominence=0.02)

    def _gen_positions(pitch: int, limit: int, proj_smooth: np.ndarray, peaks: np.ndarray,
                   search_frac: float = 0.35, min_sep_frac: float = 0.60, nbins: int = 64):
        """
        Generate grid-line positions that align with evidence in proj_smooth.
        - pitch: estimated period
        - limit: roi_w or roi_h
        - proj_smooth: smoothed 1D projection (sx or sy, NOT normalized is fine)
        - peaks: peak indices from find_peaks on the normalized projection (or on proj_smooth)
        - search_frac: window half-width as fraction of pitch for snapping
        - min_sep_frac: enforce minimum separation between output lines
        - nbins: histogram bins for phase voting
        """
        if pitch is None or pitch <= 1 or limit <= 1:
            return []

        pitch = float(pitch)
        proj_smooth = np.asarray(proj_smooth, dtype=float)
        peaks = np.asarray(peaks, dtype=int) if peaks is not None else np.array([], dtype=int)

        # 1) Estimate phase using ALL peaks (vote on peak % pitch)
        if peaks.size > 0:
            phases = np.mod(peaks.astype(float), pitch)  # [0, pitch)
            weights = proj_smooth[np.clip(peaks, 0, limit - 1)]
            bins = np.linspace(0.0, pitch, nbins + 1)
            hist, edges = np.histogram(phases, bins=bins, weights=weights)
            k = int(np.argmax(hist))
            phase = 0.5 * (edges[k] + edges[k + 1])  # phase in [0,pitch)
        else:
            # fallback: strongest point in projection
            phase = float(int(np.argmax(proj_smooth))) % pitch if proj_smooth.size else 0.0

        # 2) Generate predicted positions from phase
        n0 = int(np.floor((0.0 - phase) / pitch))
        n1 = int(np.ceil(((limit - 1) - phase) / pitch))

        w = int(max(2, round(search_frac * pitch)))
        min_sep = int(max(1, round(min_sep_frac * pitch)))

        out = []
        last = -10**9

        for n in range(n0, n1 + 1):
            pred = phase + n * pitch
            if pred < -w or pred > (limit - 1 + w):
                continue

            lo = max(0, int(round(pred)) - w)
            hi = min(limit - 1, int(round(pred)) + w)

            # 3) Snap each predicted line to evidence
            if peaks.size > 0:
                inwin = peaks[(peaks >= lo) & (peaks <= hi)]
            else:
                inwin = np.array([], dtype=int)

            if inwin.size > 0:
                best = int(inwin[np.argmax(proj_smooth[inwin])])
            else:
                # if no explicit peak in window, snap to local max of projection
                best = int(lo + np.argmax(proj_smooth[lo:hi + 1])) if hi >= lo else int(round(pred))

            # enforce monotonicity + minimum separation
            if best - last < min_sep:
                continue

            out.append(best)
            last = best

        return sorted(set(out))

    gx = _gen_positions(int(per_x) if per_x is not None else None, roi_w, sx, px)
    gy = _gen_positions(int(per_y) if per_y is not None else None, roi_h, sy, py)

    edge_margin_x = max(8, int(0.35 * per_x)) if per_x is not None else 12
    edge_margin_y = max(8, int(0.35 * per_y)) if per_y is not None else 12

    gx = sorted(set([int(v) for v in gx if edge_margin_x <= int(v) <= roi_w - 1 - edge_margin_x]))
    gy = sorted(set([int(v) for v in gy if edge_margin_y <= int(v) <= roi_h - 1 - edge_margin_y]))
        
    if roi_w > 1:
        gx = sorted(set([0] + gx + [roi_w - 1]))
    else:
        gx = [0]

    if roi_h > 1:
        gy = sorted(set([0] + gy + [roi_h - 1]))
    else:
        gy = [0]
        
    # Build debug masks (full image space)
    # lattice_full = np.zeros((H, W), dtype=np.uint8)
    # th = 6
    # for x in gx:
    #     xx = int(round(x0 + float(x)))
    #     cv2.line(lattice_full, (xx, y0), (xx, H - 1), 255, th)
    # for y in gy:
    #     yy = int(round(y0 + float(y)))
    #     cv2.line(lattice_full, (x0, yy), (W - 1, yy), 255, th)

    lattice_full = np.zeros((H, W), dtype=np.uint8)
    th = 6

    # # internal lines only
    # gx_internal = [g for g in gx if 0 < g < W - 1]
    # gy_internal = [g for g in gy if 0 < g < H - 1]

    # draw internal inferred lines
    for x in gx:
        xx = int(round(float(x)))
        cv2.line(lattice_full, (xx, 0), (xx, H - 1), 255, th)

    for y in gy:
        yy = int(round(float(y)))
        cv2.line(lattice_full, (0, yy), (W - 1, yy), 255, th)

    # draw true outer borders once
    cv2.line(lattice_full, (0, 0), (0, H - 1), 255, th)
    cv2.line(lattice_full, (W - 1, 0), (W - 1, H - 1), 255, th)
    cv2.line(lattice_full, (0, 0), (W - 1, 0), 255, th)
    cv2.line(lattice_full, (0, H - 1), (W - 1, H - 1), 255, th)

    completed_full = cv2.bitwise_or(lines_u8, lattice_full)

    if debug_dir is not None and debug_prefix:
        dd = Path(debug_dir)
        dd.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_longlines_raw_dilated.jpg"), lines_u8)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_lattice_raw.jpg"), lattice_full)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_completed_mask_raw.jpg"), completed_full)

        # White-paint overlay (mask pixels -> white) on the warped tray
        completed_overlay = warped_bgr.copy()
        completed_overlay[completed_full > 0] = (255, 255, 255)
        cv2.imwrite(str(dd / f"{debug_prefix}_sep_completed_overlay_raw.jpg"), completed_overlay)

        cols = (len(gx) - 1) if len(gx) >= 2 else None
        rows = int(round(float(roi_h) / float(per_y))) if per_y is not None and per_y > 1 else None

        overlay = warped_bgr.copy()
        # Draw synthesized periodic grid lines (covers occluded/missed separators)
        for x in gx:
            xx = int(round(x0 + float(x)))
            cv2.line(overlay, (xx, 0), (xx, H - 1), (0, 0, 255), 2)
        for y in gy:
            yy = int(round(y0 + float(y)))
            cv2.line(overlay, (0, yy), (W - 1, yy), (0, 0, 255), 2)

        if debug_dir is not None and debug_prefix:
            cv2.imwrite(str(Path(debug_dir) / f"{debug_prefix}_grid_overlay_sep.jpg"), overlay)

        info = {
            "method": "separator_longlines_raw_autocorr_lattice",
            "rim_frac": float(rim_frac),
            "roi_w": int(roi_w),
            "roi_h": int(roi_h),
            "n_peaks_x": int(len(px)),
            "n_peaks_y": int(len(py)),
            "per_x": None if per_x is None else float(per_x),
            "per_y": None if per_y is None else float(per_y),
            "n_grid_x": int(len(gx)),
            "n_grid_y": int(len(gy)),
            "grid_x": [int(v) for v in gx],
            "grid_y": [int(v) for v in gy],
            "rows": None if rows is None else int(rows),
            "cols": None if cols is None else int(cols),
            "reason": "ok" if (rows is not None and cols is not None) else "insufficient_periodicity",
        }
        return rows, cols, info, overlay

def crop_cells_from_grid(
    img_bgr: np.ndarray,
    grid_x,
    grid_y,
    out_dir,
    pad: int = 0,
    min_size: int = 8,
    prefix: str = "cell",
) -> list:
    """Crop each cell defined by consecutive vertical/horizontal grid lines.

    Args:
        img_bgr: image to crop from (typically the warped tray ROI).
        grid_x, grid_y: 1D sequences of x/y line positions in image coordinates.
        out_dir: directory to write crops into.
        pad: optional padding to shrink the crop inside each cell (positive reduces crop).
        min_size: skip crops smaller than this in either dimension.
        prefix: filename prefix.

    Returns:
        list of (r, c, path) for saved crops.
    """
    H, W = img_bgr.shape[:2]
    xs = sorted({int(round(x)) for x in grid_x if x is not None})
    ys = sorted({int(round(y)) for y in grid_y if y is not None})

    # clamp
    xs = [max(0, min(W - 1, x)) for x in xs]
    ys = [max(0, min(H - 1, y)) for y in ys]

    # ensure monotonic unique after clamp
    xs = sorted(set(xs))
    ys = sorted(set(ys))

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    for r in range(len(ys) - 1):
        y1, y2 = ys[r], ys[r + 1]
        if y2 <= y1:
            continue
        for c in range(len(xs) - 1):
            x1, x2 = xs[c], xs[c + 1]
            if x2 <= x1:
                continue

            yy1 = y1 + pad
            yy2 = y2 - pad
            xx1 = x1 + pad
            xx2 = x2 - pad

            if yy2 <= yy1 or xx2 <= xx1:
                continue
            if (yy2 - yy1) < min_size or (xx2 - xx1) < min_size:
                continue

            crop = img_bgr[yy1:yy2, xx1:xx2]
            out_path = out_dir / f"{prefix}_r{r:02d}_c{c:02d}.jpg"
            cv2.imwrite(str(out_path), crop)
            saved.append((r, c, str(out_path)))
    return saved


In [ ]:
# --- Run on full-res images (resize->YOLO mask on 640->warp full-res->trim border->obliquity correction->HLS segmentation->grid inference) ---
orig_paths = sorted(list(ORIG_DIR.glob("*.*")))
print("Full-res images found:", len(orig_paths))
orig_paths = [p for p in orig_paths if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
results = []

for p in orig_paths:
    fname = Path(p).name
    stem = Path(p).stem
    img_debug_dir = OUTDIR/'debug'/stem
    img_debug_dir.mkdir(parents=True, exist_ok=True)

    warped = cv2.imread(p)
    if warped is None:
        print("Skip unreadable:", fname)
        continue

    # 2d) Build separator evidence mask (used for tilt correction + grid inference)
    sep_mask_u8 = separator_mask_graytray_refined(warped, debug_dir=img_debug_dir, debug_prefix=stem)
    evidence_bgr = cv2.cvtColor(sep_mask_u8, cv2.COLOR_GRAY2BGR)

    out_warp_path = OUTDIR / "warped" / fname
    cv2.imwrite(str(out_warp_path), warped)

    # 5) Infer grid (rows/cols)
    sep_rows, sep_cols, sep_info, sep_overlay = infer_grid_from_separators(
        warped, rim_frac=RIM_FRAC, debug_dir=img_debug_dir, debug_prefix=stem
    )
    rows, cols, info = sep_rows, sep_cols, dict(sep_info)
    info["evidence"] = "separator_graytray"

    # 5b) Crop each cell using inferred grid lines (saved into debug/<stem>/cells/)
    try:
        if isinstance(info, dict) and ("grid_x" in info) and ("grid_y" in info):
            crops_dir = img_debug_dir / "cells"
            crop_cells_from_grid(
                warped,
                info["grid_x"],
                info["grid_y"],
                crops_dir,
                pad=0,
                min_size=8,
                prefix=stem,
            )
    except Exception as e:
        print(f"[WARN] crop_cells_from_grid failed for {stem}: {e}")

    results.append({
        "file": fname,
        "rows": None if rows is None else int(rows),
        "cols": None if cols is None else int(cols),
        "per_x": info.get("per_x"),
        "per_y": info.get("per_y"),
    })

# Save results table
import csv
csv_path = OUTDIR / "grid_counts.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()) if results else ["file"])
    w.writeheader()
    for r in results:
        w.writerow(r)

print("Done. Outputs written to:", OUTDIR)
print("  resized   ->", RESIZED_DIR)
print("  warped    ->", OUTDIR/"warped")
print("  debug     ->", OUTDIR/"debug")
print("  grid csv  ->", csv_path)
